In [162]:
import polars as pl

df = pl.read_csv(
    "../data/megascale.csv",
    null_values=["NA"],
    infer_schema_length=10000
)

columns_to_convert = ['dG_ML', 'ddG_ML']

df = df.with_columns(
    pl.col(columns_to_convert)
    .str.replace_all(r"[^0-9.-]", "")
    .cast(pl.Float64, strict=False)
)

df = df.select([
    "WT_name",
    "name",
    "deltaG",
    "mut_type",
    "aa_seq",
    "aa_seq_full",
])

df = df.rename(
    {
        'aa_seq': 'mutated_seq_short',
        'aa_seq_full': 'mutated_seq_full'
    })

df

WT_name,name,deltaG,mut_type,mutated_seq_short,mutated_seq_full
str,str,f64,str,str,str
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb""",3.332126,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wtm""",3.399832,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wte""",3.315782,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wty""",3.272933,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wth""",2.570242,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGSAGGDEVTIHLGDKTIRV…"
…,…,…,…,…,…
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-2.85014,"""I32P:L40I""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-3.924992,"""I32P:L40W""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-2.365774,"""I32P:L40Y""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"


In [163]:
import polars as pl
import re
import polars as pl

df_not_wt = df.filter(pl.col('mut_type') != "wt")


def generate_mutated_sequence_full(mutated_seq_short, mut_type: str, mutated_seq_full: str) -> str:
    # Pokud je typ mutace 'wt' nebo podobný, vrátíme původní sekvenci beze změny
    if not mut_type or "wt" in mut_type.lower():
        return "-"

    original_seq = mutated_seq_full

    # find the starting index of original_seq in aa_seq_full
    index_offset = mutated_seq_full.find(mutated_seq_short)

    if ":" in mut_type:
        for i in mut_type.split(":"):
            match = re.match(r"([A-Z])(\d+)([A-Z])", i)

            if not match:
                return "-"

            original_aa, position_str, new_aa = match.groups()
            position = int(position_str)

            index = position - 1 + index_offset

            original_seq = original_seq[:index] + original_aa + original_seq[index + 1:]
        return original_seq

    match = re.match(r"([A-Z])(\d+)([A-Z])", mut_type)

    if not match:
        return "-"

    original_aa, position_str, new_aa = match.groups()
    position = int(position_str)

    index = position - 1 + index_offset

    original_seq = mutated_seq_full[:index] + original_aa + mutated_seq_full[index + 1:]

    return original_seq


df_not_wt = df_not_wt.with_columns(
    pl.struct(['mut_type', 'mutated_seq_full', "mutated_seq_short"])
    .map_elements(
        lambda row: generate_mutated_sequence_full(row['mutated_seq_short'], row['mut_type'], row['mutated_seq_full']),
        return_dtype=pl.String)
    # Výsledek uložíme do nového sloupce 'mutated_seq'
    .alias('original_seq_full'),

)

df_not_wt = df_not_wt.select(['name', 'WT_name', 'mut_type', 'mutated_seq_full', 'original_seq_full', "deltaG"])

# Přejmenování sloupce 'aa_seq' pro lepší srozumitelnost

df_not_wt

name,WT_name,mut_type,mutated_seq_full,original_seq_full,deltaG
str,str,str,str,str,f64
"""EA|run2_0325_0005.pdb_D1Q""","""EA|run2_0325_0005.pdb""","""D1Q""","""SAGGSAGGSAGGQEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.242632
"""EA|run2_0325_0005.pdb_D1E""","""EA|run2_0325_0005.pdb""","""D1E""","""SAGGSAGGSAGGEEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.247267
"""EA|run2_0325_0005.pdb_D1N""","""EA|run2_0325_0005.pdb""","""D1N""","""SAGGSAGGSAGGNEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.269113
"""EA|run2_0325_0005.pdb_D1H""","""EA|run2_0325_0005.pdb""","""D1H""","""SAGGSAGGSAGGHEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.234823
"""EA|run2_0325_0005.pdb_D1R""","""EA|run2_0325_0005.pdb""","""D1R""","""SAGGSAGGSAGGREVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",2.912039
…,…,…,…,…,…
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40I""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-2.85014
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40W""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-3.924992
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40Y""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-2.365774


In [164]:
df_wt = df.filter(pl.col('name') == pl.col('WT_name')).filter(pl.col('mut_type') == 'wt')
df_wt

WT_name,name,deltaG,mut_type,mutated_seq_short,mutated_seq_full
str,str,f64,str,str,str
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb""",3.332126,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run3_0321_0005.pdb""","""EA|run3_0321_0005.pdb""",3.322303,"""wt""","""HDVTIHAGDKTIHVHGASEEFLRIIEQAKR…","""SAGGSAGGSAGGHDVTIHAGDKTIHVHGAS…"
"""EA|run3_0525_0006.pdb""","""EA|run3_0525_0006.pdb""",3.717377,"""wt""","""SHFELRVGTITLHFDNISEELAEELEKLAK…","""SAGGSAGGSAGGSHFELRVGTITLHFDNIS…"
"""EA|run3_1140_0005.pdb""","""EA|run3_1140_0005.pdb""",3.926922,"""wt""","""FHVTIHVGDITFHIHGVSEEEVKKLEELVR…","""SAGGSAGGSAGGFHVTIHVGDITFHIHGVS…"
"""EA|run5_0050_0004.pdb""","""EA|run5_0050_0004.pdb""",4.381241,"""wt""","""TEVDLHLGDITIKLKDVSEEIVKRAKELFK…","""SAGGSAGGSAGGTEVDLHLGDITIKLKDVS…"
…,…,…,…,…,…
"""2KT8.pdb""","""2KT8.pdb""",5.364639,"""wt""","""AEKTGIVNVSSSLNVREGASTSSKVIGSLS…","""SAGGSAAEKTGIVNVSSSLNVREGASTSSK…"
"""2KRS.pdb""","""2KRS.pdb""",6.64727,"""wt""","""MQGVVKVNSALNMRSGPGSNYGVIGTLRNN…","""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…"
"""2KYB.pdb""","""2KYB.pdb""",4.953675,"""wt""","""TGIVNVSSSLNVRSSASTSSKVIGSLSGNT…","""SAGGSAGGSAGTGIVNVSSSLNVRSSASTS…"


In [165]:
df_wt = df_wt.select(['name', 'mutated_seq_full', 'deltaG'])

df_wt = df_wt.rename(
    {
        'mutated_seq_full': 'original_seq_full',
    }
)

df = df.filter(pl.col('name') != pl.col('WT_name')).filter(pl.col('mut_type') != 'wild_type')

df_wt

name,original_seq_full,deltaG
str,str,f64
"""EA|run2_0325_0005.pdb""","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.332126
"""EA|run3_0321_0005.pdb""","""SAGGSAGGSAGGHDVTIHAGDKTIHVHGAS…",3.322303
"""EA|run3_0525_0006.pdb""","""SAGGSAGGSAGGSHFELRVGTITLHFDNIS…",3.717377
"""EA|run3_1140_0005.pdb""","""SAGGSAGGSAGGFHVTIHVGDITFHIHGVS…",3.926922
"""EA|run5_0050_0004.pdb""","""SAGGSAGGSAGGTEVDLHLGDITIKLKDVS…",4.381241
…,…,…
"""2KT8.pdb""","""SAGGSAAEKTGIVNVSSSLNVREGASTSSK…",5.364639
"""2KRS.pdb""","""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…",6.64727
"""2KYB.pdb""","""SAGGSAGGSAGTGIVNVSSSLNVRSSASTS…",4.953675


In [179]:

# df_joined = df_not_wt.join(df_wt, left_on='WT_name', right_on='name', how='left', suffix='_wt')
df_joined = df_not_wt.join(df_wt, left_on='original_seq_full', right_on='original_seq_full', how='left', suffix='_wt')

# group by mut_type and avg the dg
df_joined = df_joined.group_by("mutated_seq_full").agg(
    pl.col("deltaG").mean().alias("deltaG"),
    pl.col("deltaG_wt").mean().alias("deltaG_wt"),
    pl.col("original_seq_full").first().alias("original_seq_full"),
    #  pl.col("original_seq_full_wt").first().alias("original_seq_full_wt"),
    pl.col("mut_type").first().alias("mut_type"),
)

df_joined = df_joined.with_columns(
    (pl.col('deltaG') - pl.col('deltaG_wt')).alias('ddG')
)

df_joined.drop_nulls(subset=['ddG'])

df_joined = df_joined.select([

    # 'original_seq_full_wt',

    'original_seq_full', 'mutated_seq_full', 'deltaG', 'deltaG_wt', 'ddG', 'mut_type'
])

df_joined = df_joined.drop_nulls(["ddG"])
df_joined = df_joined.filter(pl.col('original_seq_full') != '-')

df_joined.write_csv("megascale_dataset_with_ddg.csv")

df_joined


original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type
str,str,f64,f64,f64,str
"""SAGGSAGGSAGGVKKMAKAIMADPNKADEV…","""SAGGSAGGSAGGVKKMAKAIMHDPNKADEV…",1.73325,2.118488,-0.385238,"""A10H"""
"""GTEDEKKIQELLKRANGDVSKAEKLAQSQG…","""GTEDEKKIQELLKRANGDVSKAEKLAGSQG…",1.31248,2.150429,-0.837949,"""Q26G"""
"""SAGGSAMAMNGTITTWFKDKGFGFIKDENG…","""SAGGSAMAMNGTITTWFKDKGFGFIKDENG…",4.293293,4.094748,0.198545,"""K55C"""
"""SAGGSAGGSAGKRYRAVYDYSAADEDEVSF…","""SAGGSAGGSAGKRYRAVYDYSAADEDEVSF…",3.398878,3.067643,0.331235,"""N27V:T39T"""
"""SAGGSAGGSAGGFHVTIHVGDITFHIHGVS…","""SAGGSAGGSAGGFTVTIHVGDITFHIHGVS…",4.68647,3.926922,0.759548,"""H2T"""
…,…,…,…,…,…
"""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIEHNLLSASKLYNNITFE…",3.094963,3.280557,-0.185594,"""S48E"""
"""SAGGSAGYKIYPYDSLIVTNSISVKLPKDV…","""SAGGSAGYKIYKYDSLIVTNSISVKLPKDV…",2.720596,3.837928,-1.117333,"""P5K"""
"""SAGGSAVTLFVALYDYEAITEDDLSFHKGE…","""SAGGSAVTLFVALYDYEDITEDDLSFHKGE…",1.956858,5.028433,-3.071576,"""A12D"""


In [177]:
# join back and campute ddg (just the difference)

# join WT_name to name in df_wt

df_joined = df_not_wt.join(df_wt, left_on='WT_name', right_on='name', how='left', suffix='_wt')
# df_joined = df_not_wt.join(df_wt, left_on='original_seq_full', right_on='original_seq_full', how='left', suffix='_wt')


# group by mut_type and avg the dg
df_joined = df_joined.group_by("mutated_seq_full").agg(
    pl.col("deltaG").mean().alias("deltaG"),
    pl.col("deltaG_wt").mean().alias("deltaG_wt"),
    pl.col("original_seq_full").first().alias("original_seq_full"),
    pl.col("original_seq_full_wt").first().alias("original_seq_full_wt"),
    pl.col("mut_type").first().alias("mut_type"),
)

df_joined = df_joined.with_columns(
    (pl.col('deltaG') - pl.col('deltaG_wt')).alias('ddG')
)

df_joined.drop_nulls(subset=['ddG'])

df_joined = df_joined.select([

    'original_seq_full_wt',
    'original_seq_full', 'mutated_seq_full', 'deltaG', 'deltaG_wt', 'ddG', 'mut_type'
])

df_joined.filter(pl.col('mut_type') != 'wild_type').filter(
    pl.col('original_seq_full_wt') == pl.col('original_seq_full'))

df_joined = df_joined.drop_nulls(["ddG"])
df_joined = df_joined.filter(pl.col('original_seq_full') != '-')


df_joined


original_seq_full_wt,original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type
str,str,str,f64,f64,f64,str
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…",3.88279,4.161053,-0.278263,"""A48Q"""
"""SAGGSAGGSMDETGKELVLALYDYQEKSPR…","""SAGGSMDETGKELVLALYDYQEKSPREVTM…","""SAGGSMDETGKELVMALYDYQEKSPREVTM…",4.041552,3.930052,0.111499,"""L10M"""
"""SAGGSAGGSAVTTYKLVINGKTLKGETTTK…","""SAGGSAVTTYKLVINGKTLKGETTTKAVDA…","""SAGGSAVTTYKLVINGKTLKGETTTKAVDH…",5.270467,5.200807,0.069659,"""A24H"""
"""SAGGMIINNLKLIREKKKISQSELAALLES…","""SAGGMIINNLKLIREKKKISQSELAALLES…","""SAGGMIINNTKLIREKKKISQSELAALLES…",1.210061,3.380225,-2.170165,"""L6T"""
"""SAGGSAGGSAGGMKVIFLKDVKGKGKKGEI…","""SAGGSAGGSAGGMKVIFLKDVKGKGKKGEI…","""SAGGSAGGSAGGMKVIFLKDVKGKGKKGEI…",3.349693,3.410379,-0.060686,"""K45M"""
…,…,…,…,…,…,…
"""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…","""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…","""SAGGMQGVVKVNSALNMRSGPASNYGVIGT…",0.753659,2.44228,-1.688621,"""G18A"""
"""SAGGSAGGSAMVIEDPKTNKLIEVEKLPDG…","""SAGGSAGGSAMVIEDPKTNKLIEVEKLPDG…","""SAGGSAGGSAMVIMDPKTNKLIEVEKLPDG…",3.999067,4.436625,-0.437559,"""E4M"""
"""SAGGSAGGSAGGSAGTLDMDAVLSDFVRST…","""SAGGSAGGSAGGSAGTLDMDAVLSDFVRST…","""SAGGSAGGSAGGSAGTLDMDAVLSDFVRST…",1.704465,1.642966,0.061499,"""Q42H"""


original_seq_full_wt,original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type
str,str,str,f64,f64,f64,str
"""SAGGSAGGMVQRGSKVRILRPESYWFQDVG…","""SAGGSAGGMVQRGSKVRILRPESYWFQDVG…","""SAGGSAGGMVQRGSKVRILRPESYWFQDVG…",-0.782446,2.280799,-3.063245,"""F53F:L58K"""
"""SAGGSEFTQISGYVNAFGSQRGSVLTVKVE…","""SAGGSEFTQISGYVNAFGSQRGSVLTVKVE…","""SAGGSEFTQISGYVNAFGSQRGSVLTVKVE…",-2.417792,1.703911,-4.121703,"""F37N:V52L"""
"""SAGGSAGGMATADDFKLIRDIHSTGGRRQV…","""SAGGMATADDFKLIRDIHSTGGRRQVFGSR…","""SAGGMATADDFKLIRDIHSTGGRRQVFGSR…",-0.802064,4.083619,-4.885683,"""E32P:R42M"""
"""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…","""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…","""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…",3.409993,5.264723,-1.85473,"""D46S:T49A"""
"""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIEHNLLSASKLYNNITFE…",0.553068,3.280557,-2.727489,"""Q49P:R55Y"""
…,…,…,…,…,…,…
"""SASIVTKIISIVQERQNMDDGAPVKTRDIA…","""SASIVTKIISIVQERQNMDDGAPVKTRDIA…","""SASIVTKIISIVQERQNMDDGAPVKTRDIA…",2.768125,3.288587,-0.520462,"""E43V:K53H"""
"""SAGGSAGGSAGGKDKDLLKGLDQEQANEVI…","""SAGGSAGGSAGGKDKDLLKGLDQEQANEVI…","""SAGGSAGGSAGGKDKDLLKGLDQEQANEVI…",-0.090566,2.642788,-2.733354,"""H24H:W53R"""
"""SAGMEQGTVKWFNAEKGFGFIERENGDDVF…","""SAGMEQGTVKWFNAEKGFGFIERENGDDVF…","""SAGMEQGTVKWFNAEKGFGFIERENGDDVF…",1.847002,4.794167,-2.947165,"""H29T:S31V"""
